# Scénarios de prix IPCH

À utiliser après le gel, le test final et la commande `forecast`. Le moteur applique les chocs aux indices des pays et catégories, puis recalcule l'inflation globale. Les sorties enregistrées correspondent à l'extraction de vérification.

In [1]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = Path(os.environ['CACIB_PROJECT_ROOT']) if os.environ.get('CACIB_PROJECT_ROOT') else next(p for p in candidates if (p / 'finish_project.py').exists())
sys.path.insert(0, str(ROOT))
from cacib_lab.data import lab_root, read_json, load_snapshot
from cacib_lab.evaluation import load_run, metrics, MODEL_LABELS
print('Projet :', ROOT)


Projet : /Users/hadisakho/PycharmProjects/cacib-hicp


In [2]:
from cacib_lab.workflow import unpack_scenario
from cacib_lab.scenarios import Shock, simulate, decomposition
bundle = read_json(ROOT / 'deploy/dashboard.json')
print('Coupure :', bundle['metadata']['as_of'])
print('Modèle :', bundle['metadata']['selected_model'])
print('Fenêtre :', bundle['metadata']['window_months'])
data = unpack_scenario(bundle)
start = bundle['scenario']['future_months'][0]


Coupure : 2026-09
Modèle : ensemble__ets_branches
Fenêtre : 120


## Cohérence du scénario nul

In [3]:
zero = simulate(*data, [])
assert np.allclose(zero.impact_pp, 0, atol=1e-12)
np.testing.assert_allclose(zero.baseline_index, [r['predicted_index'] for r in bundle['forecast']], rtol=1e-10)
display(zero)


,target,baseline_index,scenario_index,baseline_yoy,scenario_yoy,impact_pp
0,2026-10,104.778587,104.778587,4.008921,4.008921,0.0
1,2026-11,104.648711,104.648711,4.179902,4.179902,0.0
2,2026-12,104.944469,104.944469,4.287458,4.287458,0.0


## Hausse permanente des prix de transport en France

Une hausse de 10 % des prix de transport n'est pas une hausse de 10 points de l'inflation annuelle globale.

In [4]:
shock = Shock(('FR',), ('CP07',), 10., start, 'permanent', 1)
result = simulate(*data, [shock])
display(result)
fig = go.Figure()
fig.add_scatter(x=result.target, y=result.baseline_yoy, name='Trajectoire centrale', mode='lines+markers')
fig.add_scatter(x=result.target, y=result.scenario_yoy, name='Avec choc', mode='lines+markers')
fig.update_layout(template='plotly_white', yaxis_title='Inflation annuelle (%)', title='Scénario conditionnel')
fig.show(renderer='plotly_mimetype')


,target,baseline_index,scenario_index,baseline_yoy,scenario_yoy,impact_pp
0,2026-10,104.778587,105.091952,4.008921,4.319985,0.311063
1,2026-11,104.648711,104.961138,4.179902,4.490928,0.311027
2,2026-12,104.944469,105.258545,4.287458,4.599568,0.312110


## Comparer les profils

Permanent : niveau augmenté durablement. Temporaire : retour de la catégorie à sa trajectoire centrale. Répété : supplément mensuel qui se cumule pendant la durée choisie.

In [5]:
frames = []
for kind, magnitude, duration in [('permanent', 10., 1), ('temporary', 10., 1), ('monthly', 1., 3)]:
    r = simulate(*data, [Shock(('FR',), ('CP07',), magnitude, start, kind, duration)])
    frames.append(r.assign(profile=kind))
comparison = pd.concat(frames, ignore_index=True)
display(comparison[['profile', 'target', 'impact_pp']])
fig = go.Figure()
for kind, group in comparison.groupby('profile'):
    fig.add_scatter(x=group.target, y=group.impact_pp, name=kind, mode='lines+markers')
fig.update_layout(template='plotly_white', yaxis_title='Impact (points de pourcentage)', title='Profils de choc explicitement différents')
fig.show(renderer='plotly_mimetype')


,profile,target,impact_pp
0,permanent,2026-10,0.311063
1,permanent,2026-11,0.311027
2,permanent,2026-12,0.312110
3,temporary,2026-10,0.311063
4,temporary,2026-11,0.000000
5,temporary,2026-12,0.000000
6,monthly,2026-10,0.031106
7,monthly,2026-11,0.062516
8,monthly,2026-12,0.094573


## Plusieurs pays et catégories

L'attribution est séquentielle et somme à l'impact total; en présence d'interactions elle dépend de l'ordre choisi.

In [6]:
shocks = [Shock(('FR', 'DE'), ('CP01', 'CP07'), 5., start, 'permanent', 1)]
multi = simulate(*data, shocks)
contributions = decomposition(*data, shocks)
summed = contributions.groupby('target').impact_pp.sum()
np.testing.assert_allclose(summed.to_numpy(), multi.impact_pp.to_numpy(), atol=1e-10)
display(multi)
display(contributions)


,target,baseline_index,scenario_index,baseline_yoy,scenario_yoy,impact_pp
0,2026-10,104.778587,105.510634,4.008921,4.735590,0.726669
1,2026-11,104.648711,105.380800,4.179902,4.908711,0.728809
2,2026-12,104.944469,105.678326,4.287458,5.016721,0.729263


,target,geo,component,impact_pp
0,2026-10,DE,CP01,0.179701
1,2026-11,DE,CP01,0.181213
2,2026-12,DE,CP01,0.181674
3,2026-10,DE,CP07,0.251913
4,2026-11,DE,CP07,0.251751
5,2026-12,DE,CP07,0.251021
6,2026-10,FR,CP01,0.139523
7,2026-11,FR,CP01,0.140332
8,2026-12,FR,CP01,0.140513
9,2026-10,FR,CP07,0.155532


## Export et interprétation

In [7]:
destination = ROOT / 'reports/lab/scenarios/notebook_example'
destination.mkdir(parents=True, exist_ok=True)
result.to_csv(destination / 'scenario.csv', index=False)
contributions.to_csv(destination / 'decomposition.csv', index=False)
print('Exports :', destination)


Exports : /Users/hadisakho/PycharmProjects/cacib-hicp/reports/lab/scenarios/notebook_example


Les poids et les autres trajectoires restent conditionnellement fixes. Un choc temporaire peut laisser un résidu agrégé au passage de janvier à cause du chaînage. Ces résultats ne constituent pas une estimation causale de la transmission du pétrole, des salaires ou de la politique monétaire. Les intervalles du scénario central ne sont pas présentés comme une incertitude causale du choc.